# SQUID Data Analysis

In [ ]:
import sys
sys.path.insert(0, "..")          # expose the AutoSQUID package
sys.path.append("../../")         # RanLabPythonRepo root (same as the other notebooks)
import AutoSQUID as sq
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Functions used here to analyze data:
# read_daq_file: read a raw DAQ file into header info and a dataframe
# plot_overlay: overlay the voltage trace and temperature trace on a shared-time axis
# plot_psd: plot one-sided Welch PSD (Phi_0^2/Hz) of one trace

In [ ]:
data_root = r''     # your data root
user = ''           # data folder owner
date = ''           # date subfolder, e.g. 'Jun-01-2026' ('' = the user root)
path = os.path.join(data_root, user, date)    # folder holding the DAQ_*.txt traces

# Plot Raw Data

In [ ]:
filenames = [f for f in os.listdir(path) if 'DAQ' in f and f.endswith('.txt')]

for filename in filenames:
    header_info, df = sq.read_daq_file(path, filename)
    dt = header_info["SCANINTVAL"]                       # scan interval from the header (robust vs filename parsing)
    t = np.array(df.index) * dt
    y = np.array(df["CHAN_01(V)"])
    plt.plot(t, y)
    plt.xlabel('Time (s)')
    plt.ylabel('Array Voltage (V)')
    plt.title(f'{filename}')
    plt.show()

# Plot Data-Temp Overlay

In [ ]:
filenames = [file for file in os.listdir(path) if ('DAQ' in file) and (file.endswith('.txt'))
                                            and ('100us' in file) and ('SURGE' in file)]

for filename in filenames:
    header, df = sq.read_daq_file(path, filename)
    dt = header["SCANINTVAL"]
    t  = np.array(df.index) * dt
    v  = np.array(df["CHAN_01(V)"])
    s  = max(1, len(v) // 200_000)                           # decimate the dense trace for plotting
    t_dec, v_dec = t[::s], v[::s]

    tdf = pd.read_csv(path + "\\" + filename.replace(".txt", "_temp.csv"))   # time_s, T_K
    sq.plot_overlay(t_dec, v_dec, tdf["time_s"].values, tdf["T_K"].values, title=filename)

# Plot PSD

In [ ]:
filenames = ['']
conversion = 0.762     # f0/V for this May 2026 cooldown

for filename in filenames:
    sq.plot_psd(path, filename, conversion, P=(10, 100, 1000, 10000), clean_only=True)